# Centralized Assignment of EVs to Existing Charging Stations on Swiss Highways

## Overview

This notebook implements a **centralized optimization model** that assigns electric vehicles (EVs) to existing charging stations along Swiss highways.

The study covers **three Swiss highways** in **both directions**:
- **A2**: Basel ↔ Como
- **A8**: Luzern ↔ Thun
- **A13**: Sankt Margrethen ↔ Bellinzona

### How the model works

1. **Input data** (highway rest areas + existing EV charger locations) are loaded from the `Inputs/` folder.
2. A synthetic population of **100 EVs per scenario** is generated, each with a random battery level.
3. For each EV, the model computes which charging stations are **physically reachable** before the battery runs out.
4. A **CP-SAT constraint programming solver** (Google OR-Tools) assigns each feasible EV to exactly one station, minimizing queue congestion and balancing load across stations.
5. After solving, a **queue simulation** computes the actual waiting times per car.
6. All results are exported to the `Outputs/` folder.

### Input files (must be placed in `Inputs/`)

| File | Description |
|------|-------------|
| `A2BaleComoCsv.csv` | A2 rest areas: Basel → Como |
| `A2ComoBaleCsv.csv` | A2 rest areas: Como → Basel |
| `A8LuzernThunCsv.csv` | A8 rest areas: Luzern → Thun |
| `A8ThunLuzernCsv.csv` | A8 rest areas: Thun → Luzern |
| `A13SanktMargrethenBellinzonaCsv.csv` | A13 rest areas: Sankt Margrethen → Bellinzona |
| `A13BellinzonaSanktMargrethenCsv.csv` | A13 rest areas: Bellinzona → Sankt Margrethen |
| `EV_chargeurs.csv` | Existing EV charger presence by rest area |

### Output files (saved in `Outputs/`)

For each of the 6 scenarios, the following files are generated:
- `{scenario}_generated_cars.csv` — synthetic EV population
- `{scenario}_reachability_matrix_cars_stations.csv` — which stations each car can reach
- `{scenario}_optimized_car_station_assignments.csv` — final assignment + waiting times
- `{scenario}_station_queue_summary.csv` — per-station load and waiting time summary
- `{scenario}_decision_matrix_car_station.csv` — binary assignment matrix

A global summary across all scenarios is saved as `global_scenario_summary.csv`.

---
## 1. Imports and Environment Setup

We import all required libraries and verify the Python environment. The key dependencies are:
- **NumPy / Pandas**: data manipulation
- **Matplotlib**: plotting (backend set to `Agg` for non-interactive rendering)
- **OR-Tools (`ortools.sat.python.cp_model`)**: the constraint programming solver used for the assignment optimization

A fixed random seed (`42`) is set for full reproducibility of results.

In [1]:
import os
import re
import glob
import sys
import random
from collections import defaultdict

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from ortools.sat.python import cp_model

# ── Reproducibility ──────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)

# ── Environment check ────────────────────────────────────────────────────────
print("Python executable:", sys.executable)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

Python executable: /opt/anaconda3/envs/evcharging/bin/python
NumPy version: 2.4.6
Pandas version: 3.0.3


---
## 2. Folder Configuration

We define the paths to the `Inputs/` and `Outputs/` directories. The output directory is created automatically if it does not already exist.

In [2]:
INPUT_DIR  = "Inputs"
OUTPUT_DIR = "Outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input  directory : {os.path.abspath(INPUT_DIR)}")
print(f"Output directory : {os.path.abspath(OUTPUT_DIR)}")

Input  directory : /Users/charlelie/Desktop/MA2/Projet log/Inputs
Output directory : /Users/charlelie/Desktop/MA2/Projet log/Outputs


---
## 3. Helper Functions

Three utility functions are defined here and reused throughout the notebook:

| Function | Purpose |
|----------|---------|
| `normalize_filename(filename)` | Strips special characters and lowercases a filename so that CSV files can be matched robustly regardless of casing or punctuation. |
| `extract_km(value)` | Parses a localization string such as `"PK 12 km"` and returns the numeric kilometer value (`12.0`). |
| `normalize_rest_area_name(s)` | Normalizes rest area names for cross-source matching (e.g. translates `"ouest"` → `"west"`, removes accents and punctuation). |

In [3]:
def normalize_filename(filename):
    """
    Normalize a filename to make robust matching possible.
    Removes all non-alphanumeric characters and lowercases the result.
    """
    return re.sub(r"[^a-z0-9]", "", filename.lower())


def extract_km(value):
    """
    Extract the first numeric value from a localization string.
    Example: 'PK 12 km' -> 12.0
    Returns None if no number is found or input is NaN.
    """
    if pd.isna(value):
        return None
    match = re.search(r"(\d+(?:\.\d+)?)", str(value))
    return float(match.group(1)) if match else None


def normalize_rest_area_name(s):
    """
    Normalize rest area names for matching between CSV sources.
    Translates directional words (ouest->west, est->ost, sud->sued)
    and removes all non-alphanumeric characters.
    """
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    replacements = {
        "ouest": "west",
        "est":   "ost",
        "sud":   "sued",
        "süd":   "sued",
        "nord":  "nord",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)
    s = re.sub(r"[^a-z0-9]", "", s)
    return s

---
## 4. Step 1 — Load Highway Rest-Area CSV Files

We load the six highway CSV files from the `Inputs/` directory. Each file describes the rest areas ("aires") along one highway in one direction, with their name (`Aire`) and kilometer position (`Localisation`).

**Matching strategy**: instead of relying on exact filenames, we normalize both the expected names and the actual filenames on disk (removing spaces, accents, and special characters) and match them programmatically. This makes the loading robust to minor naming variations.

The six expected files and their corresponding highway/direction:

| Key | Direction |
|-----|-----------|
| `(A2, +1)` | Basel → Como |
| `(A2, -1)` | Como → Basel |
| `(A8, +1)` | Luzern → Thun |
| `(A8, -1)` | Thun → Luzern |
| `(A13, +1)` | Sankt Margrethen → Bellinzona |
| `(A13, -1)` | Bellinzona → Sankt Margrethen |

After loading, all files are concatenated into a single DataFrame `df_aires` with columns: `autoroute`, `nom`, `pk`, `sens`.

In [4]:
print("=" * 60)
print("STEP 1 — Loading highway rest-area CSV files from Inputs/")
print("=" * 60)

all_csv = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
print("CSV files found:", all_csv)

# Expected normalized filenames mapped to (highway, direction) keys
expected_map = {
    ("A2",  +1): "a2balecomocsv",
    ("A2",  -1): "a2comobalecsv",
    ("A8",  +1): "a8luzernthuncsv",
    ("A8",  -1): "a8thunluzerncsv",
    ("A13", +1): "a13sanktmargrethenbellinzonacsv",
    ("A13", -1): "a13bellinzonasanktmargrethencsv",
}

csv_files = {}

for key, expected_norm in expected_map.items():
    matches = [
        f for f in all_csv
        if normalize_filename(os.path.basename(f)) == expected_norm
    ]
    if len(matches) == 1:
        csv_files[key] = matches[0]
    elif len(matches) == 0:
        print(f"  ⚠️  File not found for {key}, expected normalized name = {expected_norm}")
    else:
        print(f"  ⚠️  Multiple matches for {key}:", matches)

print("\nFiles matched:")
for k, v in csv_files.items():
    print(f"  {k} -> {v}")

if len(csv_files) != 6:
    raise ValueError(
        "Not all required files were found automatically. "
        "Please check the CSV filenames in the Inputs/ folder."
    )

# ── Read and concatenate all files ───────────────────────────────────────────
all_rows = []

for (autoroute, sens), path in csv_files.items():
    df = pd.read_csv(
        path,
        sep=None,
        engine="python",
        encoding="utf-8-sig",
        on_bad_lines="skip",
    )
    df.columns = df.columns.str.strip()

    print(f"\nReading {path}")
    print("  Columns detected:", list(df.columns))

    if "Aire" not in df.columns or "Localisation" not in df.columns:
        raise ValueError(
            f"File {path} must contain columns 'Aire' and 'Localisation'."
        )

    df = df[df["Aire"].notna()].copy()
    df["pk"] = df["Localisation"].apply(extract_km)
    df = df[df["pk"].notna()].copy()

    df["autoroute"]           = autoroute
    df["nom"]                 = df["Aire"].astype(str).str.strip()
    df["sens"]                = sens
    df["has_existing_charger"] = False
    df["existing_chargers"]   = 0

    df = df[["autoroute", "nom", "pk", "sens", "has_existing_charger", "existing_chargers"]]
    all_rows.append(df)

df_aires = pd.concat(all_rows, ignore_index=True)
df_aires = df_aires.sort_values(["autoroute", "sens", "pk"]).reset_index(drop=True)
df_aires["id"] = range(len(df_aires))

print("\nFirst rows of the rest-area DataFrame:")
print(df_aires.head().to_string(index=False))
print(f"\nTotal number of rest areas: {len(df_aires)}")

STEP 1 — Loading highway rest-area CSV files from Inputs/
CSV files found: ['Inputs/A13_Sankt-Margrethen_Bellinzona.csv', 'Inputs/A2_Como_Bale.csv', 'Inputs/A8_Luzern_Thun.csv', 'Inputs/decision_matrix_car_station.csv', 'Inputs/A8_Thun_Luzern.csv', 'Inputs/A2_Bale_Como.csv', 'Inputs/A13_Bellinzona _Sankt-Margrethen.csv', 'Inputs/EV_chargeurs.csv']

Files matched:
  ('A2', 1) -> Inputs/A2_Bale_Como.csv
  ('A2', -1) -> Inputs/A2_Como_Bale.csv
  ('A8', 1) -> Inputs/A8_Luzern_Thun.csv
  ('A8', -1) -> Inputs/A8_Thun_Luzern.csv
  ('A13', 1) -> Inputs/A13_Sankt-Margrethen_Bellinzona.csv
  ('A13', -1) -> Inputs/A13_Bellinzona _Sankt-Margrethen.csv

Reading Inputs/A2_Bale_Como.csv
  Columns detected: ['Aire', 'Type', 'Localisation', 'Équipements aire de repos', 'Carburant', 'Restauration', 'Hôtellerie', 'Autres', 'Commentaire', 'Site internet']

Reading Inputs/A2_Como_Bale.csv
  Columns detected: ['Aire', 'Type', 'Localisation', 'Équipements aire de repos', 'Carburant', 'Restauration', 'Hôtelle

---
## 5. Step 2 — Enrich with Existing EV Charger Data

We now load `Inputs/EV_chargeurs.csv`, which indicates which rest areas already have EV charging infrastructure. This file is merged with `df_aires` using normalized highway and rest-area name keys.

Additionally, a `provider_map` dictionary encodes **manually curated knowledge** about:
- the charging provider(s) present at each station (e.g. IONITY, GoFast, Move)
- the number of known chargers at each station

After enrichment, each rest area in `df_aires` has the following new columns:

| Column | Description |
|--------|-------------|
| `has_existing_charger` | Boolean — whether the station already has chargers |
| `existing_chargers` | Number of chargers (from `provider_map`) |
| `ev_providers` | Provider name(s) (e.g. `"IONITY"`, `"GoFast + IONITY"`) |
| `ev_charger_presence` | Human-readable `"Oui"` / `"Non"` |
| `existing_chargers_known` | Charger count, or `NaN` if presence confirmed but count unknown |

In [5]:
print("=" * 60)
print("STEP 2 — Enriching with Inputs/EV_chargeurs.csv")
print("=" * 60)

ev_file = os.path.join(INPUT_DIR, "EV_chargeurs.csv")

if not os.path.exists(ev_file):
    raise FileNotFoundError(f"File not found: {ev_file}")

ev_template = pd.read_csv(ev_file, sep=";", encoding="utf-8-sig")
ev_template = ev_template.drop(
    columns=[col for col in ev_template.columns if "Unnamed" in col],
    errors="ignore",
)

required_ev_cols = ["Highway", "Rest stop name", "EV Chargeur"]
for col in required_ev_cols:
    if col not in ev_template.columns:
        raise ValueError(f"Column '{col}' is missing from {ev_file}.")

ev_template["highway_norm"] = ev_template["Highway"].astype(str).str.strip().str.upper()
ev_template["name_norm"]    = ev_template["Rest stop name"].apply(normalize_rest_area_name)
ev_template["ev_present_csv"] = ev_template["EV Chargeur"].fillna(0).astype(int)

# ── Provider map: manually curated charger counts and provider names ──────────
provider_map = {
    ("A2",  "baselweil"):        {"providers": "",                  "known_chargers": 4},
    ("A2",  "pratteln"):         {"providers": "IONITY",             "known_chargers": 6},
    ("A2",  "neuenkirchwest"):   {"providers": "GoFast",             "known_chargers": 4},
    ("A2",  "neuenkirchost"):    {"providers": "GoFast",             "known_chargers": 4},
    ("A2",  "gotthard"):         {"providers": "IONITY",             "known_chargers": 6},
    ("A2",  "quinto"):           {"providers": "GoFast",             "known_chargers": 5},
    ("A2",  "stalvedro"):        {"providers": "GoFast",             "known_chargers": 5},
    ("A2",  "bellinzonanord"):   {"providers": "Move",               "known_chargers": 4},
    ("A2",  "bellinzonasued"):   {"providers": "Move",               "known_chargers": 4},
    ("A2",  "bellinzonasud"):    {"providers": "Move",               "known_chargers": 4},
    ("A2",  "coldreriowest"):    {"providers": "GoFast",             "known_chargers": 7},
    ("A2",  "coldrerioost"):     {"providers": "GoFast",             "known_chargers": 5},
    ("A2",  "coldrerioest"):     {"providers": "GoFast",             "known_chargers": 5},
    ("A8",  "iseltwaldrepos"):   {"providers": "",                  "known_chargers": 6},
    ("A13", "rheintal"):         {"providers": "Move",               "known_chargers": 4},
    ("A13", "heidiland"):        {"providers": "GoFast + IONITY",    "known_chargers": 6},
    ("A13", "viamala"):          {"providers": "GoFast",             "known_chargers": 16},
}

# ── Merge charger presence into df_aires ─────────────────────────────────────
df_aires["highway_norm"] = df_aires["autoroute"].astype(str).str.strip().str.upper()
df_aires["name_norm"]    = df_aires["nom"].apply(normalize_rest_area_name)

df_aires = df_aires.merge(
    ev_template[["highway_norm", "name_norm", "ev_present_csv"]],
    on=["highway_norm", "name_norm"],
    how="left",
)
df_aires["ev_present_csv"] = df_aires["ev_present_csv"].fillna(0).astype(int)


def get_provider_info(row):
    """Look up provider name and charger count from the provider_map."""
    key = (row["highway_norm"], row["name_norm"])
    return provider_map.get(key, {"providers": "", "known_chargers": 0})


provider_info = df_aires.apply(get_provider_info, axis=1)
df_aires["ev_providers"]           = provider_info.apply(lambda x: x["providers"])
df_aires["known_chargers_comments"] = provider_info.apply(lambda x: x["known_chargers"])

df_aires["has_existing_charger"] = df_aires["ev_present_csv"].astype(bool)

# Clear provider info for stations without chargers
df_aires.loc[~df_aires["has_existing_charger"], "ev_providers"]           = ""
df_aires.loc[~df_aires["has_existing_charger"], "known_chargers_comments"] = 0

df_aires["existing_chargers"] = df_aires["known_chargers_comments"]

df_aires = df_aires.drop(columns=["highway_norm", "name_norm"], errors="ignore")

# ── Human-readable presence column ───────────────────────────────────────────
df_aires["ev_charger_presence"]   = df_aires["has_existing_charger"].map({True: "Oui", False: "Non"})
df_aires["existing_chargers_known"] = df_aires["known_chargers_comments"]

df_aires.loc[df_aires["ev_charger_presence"] == "Non", "existing_chargers_known"] = 0
df_aires.loc[
    (df_aires["ev_charger_presence"] == "Oui") & (df_aires["existing_chargers_known"] == 0),
    "existing_chargers_known",
] = pd.NA

print(
    df_aires[
        ["autoroute", "nom", "sens", "has_existing_charger", "existing_chargers", "ev_providers"]
    ]
    .head(30)
    .to_string(index=False)
)

STEP 2 — Enriching with Inputs/EV_chargeurs.csv
autoroute                        nom  sens  has_existing_charger  existing_chargers    ev_providers
      A13              Kriessern-Süd    -1                 False                  0                
      A13               Rheintal-Ost    -1                 False                  0                
      A13                  Heidiland    -1                  True                  6 GoFast + IONITY
      A13                  Apfelwuhr    -1                 False                  0                
      A13                    Viamala    -1                  True                 16          GoFast
      A13                      Rofla    -1                 False                  0                
      A13                      Isola    -1                 False                  0                
      A13                     Ghiffa    -1                 False                  0                
      A13             Campagnola-Sud    -1          

---
## 6. Step 3 — Global Scenario Parameters

We define the parameters that govern the simulation and optimization for all scenarios. These values are shared across all six highway/direction combinations.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `N_CARS` | 100 | Number of EVs generated per scenario |
| `ENTRY_FLOW_MIN` | 1 | One car enters the highway every 1 minute |
| `SPEED_KMH` | 100 | Average highway driving speed |
| `AUTONOMIE_MAX_KM` | 400 | Maximum range of EVs on a full battery |
| `BATTERY_MEAN` | 0.50 | Mean initial battery level (50%) |
| `BATTERY_STD` | 0.18 | Standard deviation of initial battery level |
| `BATTERY_MIN/MAX` | 0.10 / 0.90 | Battery level clipping bounds |
| `RECHARGE_TIME_MIN` | 10 | Fixed charging session duration (minutes) |
| `MAX_QUEUE_PER_STATION` | 3 | Soft queue threshold — excess is penalized in the objective |
| `SOLVER_TIME_LIMIT_S` | 30 | Maximum solver runtime per scenario (seconds) |

In [6]:
print("=" * 60)
print("STEP 3 — Global scenario parameters")
print("=" * 60)

# ── Scenarios ─────────────────────────────────────────────────────────────────
SCENARIOS = [
    ("A2",  1),
    ("A2", -1),
    ("A8",  1),
    ("A8", -1),
    ("A13",  1),
    ("A13", -1),
]

# ── Traffic parameters ────────────────────────────────────────────────────────
N_CARS          = 100
ENTRY_FLOW_MIN  = 1        # one car enters every N minutes

# ── EV physical parameters ────────────────────────────────────────────────────
SPEED_KMH        = 100
AUTONOMIE_MAX_KM = 400

BATTERY_MEAN = 0.50
BATTERY_STD  = 0.18
BATTERY_MIN  = 0.10
BATTERY_MAX  = 0.90

# ── Station / queue parameters ────────────────────────────────────────────────
RECHARGE_TIME_MIN      = 15
MAX_QUEUE_PER_STATION  = 2

# ── Solver time limit ─────────────────────────────────────────────────────────
SOLVER_TIME_LIMIT_S = 30.0

print("Scenarios to be studied:")
for highway, sens in SCENARIOS:
    direction = "→" if sens == 1 else "←"
    print(f"  {highway}  direction={sens} ({direction})")

print(f"\nNumber of EVs per scenario  : {N_CARS}")
print(f"Entry flow                  : 1 car every {ENTRY_FLOW_MIN} min")
print(f"Charging time               : {RECHARGE_TIME_MIN} min")
print(f"Max desired queue           : {MAX_QUEUE_PER_STATION} car(s)")

STEP 3 — Global scenario parameters
Scenarios to be studied:
  A2  direction=1 (→)
  A2  direction=-1 (←)
  A8  direction=1 (→)
  A8  direction=-1 (←)
  A13  direction=1 (→)
  A13  direction=-1 (←)

Number of EVs per scenario  : 100
Entry flow                  : 1 car every 1 min
Charging time               : 15 min
Max desired queue           : 2 car(s)


---
## 7. Step 4 — Scenario Function

The `run_scenario()` function encapsulates the full pipeline for a single highway/direction pair. It is called once per scenario in Step 5.

### Internal pipeline

#### 4a. Station selection
Filters `df_aires` to keep only rest areas on the target highway and direction that already have EV chargers. Stations are ordered by kilometer position in the direction of travel.

#### 4b. EV generation
Generates `N_CARS` synthetic EVs. Each car:
- enters the highway at minute `car_id × ENTRY_FLOW_MIN`
- has a battery level drawn from a clipped normal distribution
- has a remaining range = `battery_pct × AUTONOMIE_MAX_KM`

#### 4c. Reachability matrix
For each (car, station) pair, we check two conditions:
1. The car can **reach** the station before running out of battery.
2. After charging, the car can **reach the highway exit** (full battery assumed after charging).

Only feasible (car, station) pairs are passed to the solver.

#### 4d. CP-SAT optimization model
Variables:
- **`y[car_id, station_id]`** — binary: 1 if car `car_id` is assigned to `station_id`

Constraints:
- Each feasible car is assigned to **exactly one** station.

Soft queue modeling:
- Cars are grouped into **time slots** of `RECHARGE_TIME_MIN` minutes per station.
- Queue overflow beyond `MAX_QUEUE_PER_STATION` is captured in `over_queue_vars`.

Objective (minimized, weighted sum):

| Term | Weight | Purpose |
|------|--------|---------|
| Over-queue penalty | 100,000 | Strongly penalize congestion spikes |
| Queue penalty | 10,000 | Penalize any queue |
| Load balance deviation | 500 | Spread cars evenly across stations |
| Max station load | 100 | Limit the busiest station |
| Early-stop penalty | 1 | Prefer sending cars to later stations when possible |

#### 4e. Queue simulation
After solving, a deterministic FIFO queue simulation computes the **actual waiting time** for each assigned car based on charger availability at its assigned station.

#### 4f. Output files
Five CSV files are written to `Outputs/` for each scenario.

In [7]:
def run_scenario(selected_highway, selected_sens, scenario_id):
    """
    Run the centralized EV-to-station assignment model for one highway and one direction.

    Parameters
    ----------
    selected_highway : str
        Highway identifier, e.g. 'A2', 'A8', 'A13'.
    selected_sens : int
        Direction: +1 (increasing km) or -1 (decreasing km).
    scenario_id : int
        Integer index used to offset the random seed for reproducibility.

    Returns
    -------
    dict
        Summary metrics for this scenario (feasibility, waiting times, etc.).
    """

    print("\n" + "#" * 80)
    print(f"SCENARIO {scenario_id} — {selected_highway}, direction={selected_sens}")
    print("#" * 80)

    # ── Output file prefix ────────────────────────────────────────────────────
    sens_label    = "sens_plus" if selected_sens == 1 else "sens_minus"
    output_prefix = f"{selected_highway}_{sens_label}"

    # ── Prepare local copy of df_aires with charger count column ─────────────
    local_df_aires = df_aires.copy()
    local_df_aires["existing_chargers_model"] = 0

    if "known_chargers_comments" in local_df_aires.columns:
        local_df_aires["existing_chargers_model"] = (
            local_df_aires["known_chargers_comments"].fillna(0)
        )
    elif "existing_chargers" in local_df_aires.columns:
        local_df_aires["existing_chargers_model"] = (
            local_df_aires["existing_chargers"].fillna(0)
        )

    # If a station has chargers but count is 0, assume at least 1
    local_df_aires.loc[
        local_df_aires["has_existing_charger"] & (local_df_aires["existing_chargers_model"] <= 0),
        "existing_chargers_model",
    ] = 1
    local_df_aires.loc[
        ~local_df_aires["has_existing_charger"],
        "existing_chargers_model",
    ] = 0
    local_df_aires["existing_chargers_model"] = local_df_aires["existing_chargers_model"].astype(int)

    # ── 4a. Select existing stations on the target axis ───────────────────────
    df_stations = local_df_aires[
        (local_df_aires["autoroute"]           == selected_highway)
        & (local_df_aires["sens"]              == selected_sens)
        & (local_df_aires["has_existing_charger"])
    ].copy()

    if df_stations.empty:
        print(f"  ⚠️  No existing stations for {selected_highway}, direction={selected_sens}. Skipping.")
        return {
            "scenario": output_prefix, "highway": selected_highway, "sens": selected_sens,
            "status": "NO_STATION", "n_cars": N_CARS, "feasible_cars": 0,
            "infeasible_cars": N_CARS, "assigned_cars": 0,
            "total_waiting_time_min": None, "average_waiting_time_min": None,
            "max_waiting_time_min": None,
        }

    ascending = selected_sens == 1
    df_stations = df_stations.sort_values("pk", ascending=ascending).reset_index(drop=True)
    df_stations["station_id"] = range(len(df_stations))

    stations   = df_stations.to_dict("records")
    m_stations = len(stations)

    print(f"\nExisting stations retained: {m_stations}")
    print(
        df_stations[
            ["station_id", "autoroute", "nom", "sens", "pk", "existing_chargers_model", "ev_providers"]
        ].to_string(index=False)
    )

    # ── Entry / exit km ───────────────────────────────────────────────────────
    df_axis = local_df_aires[
        (local_df_aires["autoroute"] == selected_highway)
        & (local_df_aires["sens"]    == selected_sens)
    ].copy()

    pk_min = float(df_axis["pk"].min())
    pk_max = float(df_axis["pk"].max())

    entry_pk      = pk_min if selected_sens == 1 else pk_max
    exit_pk       = pk_max if selected_sens == 1 else pk_min
    axis_length_km = abs(exit_pk - entry_pk)

    print(f"\nEntry PK     : {entry_pk:.1f} km")
    print(f"Exit PK      : {exit_pk:.1f} km")
    print(f"Axis length  : {axis_length_km:.1f} km")

    # ── 4b. Generate EV population ────────────────────────────────────────────
    rng           = np.random.default_rng(42 + scenario_id)
    battery_levels = np.clip(
        rng.normal(loc=BATTERY_MEAN, scale=BATTERY_STD, size=N_CARS),
        BATTERY_MIN,
        BATTERY_MAX,
    )

    cars = [
        {
            "car_id":           car_id,
            "entry_time_min":   car_id * ENTRY_FLOW_MIN,
            "entry_pk":         entry_pk,
            "exit_pk":          exit_pk,
            "battery_pct":      float(battery_levels[car_id]),
            "remaining_range_km": float(battery_levels[car_id]) * AUTONOMIE_MAX_KM,
        }
        for car_id in range(N_CARS)
    ]

    df_cars = pd.DataFrame(cars)
    df_cars.to_csv(
        os.path.join(OUTPUT_DIR, f"{output_prefix}_generated_cars.csv"),
        index=False, encoding="utf-8-sig",
    )

    print("\nFirst 10 generated cars:")
    print(df_cars.head(10).to_string(index=False))

    # ── 4c. Build reachability matrix ─────────────────────────────────────────
    reachability               = {}
    feasible_stations_by_car   = defaultdict(list)

    for car in cars:
        car_id          = car["car_id"]
        remaining_range = car["remaining_range_km"]
        entry_time_min  = car["entry_time_min"]

        for st in stations:
            station_id  = st["station_id"]
            station_pk  = float(st["pk"])

            if selected_sens == 1:
                distance_to_station = station_pk - entry_pk
                station_after_entry = station_pk >= entry_pk
                distance_station_to_exit = exit_pk - station_pk
            else:
                distance_to_station = entry_pk - station_pk
                station_after_entry = station_pk <= entry_pk
                distance_station_to_exit = station_pk - exit_pk

            if not station_after_entry:
                continue

            can_reach_station      = distance_to_station      <= remaining_range
            can_reach_exit_after   = distance_station_to_exit <= AUTONOMIE_MAX_KM

            if can_reach_station and can_reach_exit_after:
                travel_time_min  = distance_to_station / SPEED_KMH * 60
                arrival_time_min = entry_time_min + travel_time_min
                slot             = int(arrival_time_min // RECHARGE_TIME_MIN)
                battery_slack_km = remaining_range - distance_to_station

                reachability[(car_id, station_id)] = {
                    "distance_to_station_km":    distance_to_station,
                    "distance_station_to_exit_km": distance_station_to_exit,
                    "arrival_time_min":           arrival_time_min,
                    "slot":                       slot,
                    "battery_slack_km":           battery_slack_km,
                }
                feasible_stations_by_car[car_id].append(station_id)

    infeasible_cars = [c["car_id"] for c in cars if len(feasible_stations_by_car[c["car_id"]]) == 0]
    feasible_cars   = [c for c in cars if c["car_id"] not in infeasible_cars]

    print(f"\nTotal cars                        : {N_CARS}")
    print(f"Cars with at least one valid station: {len(feasible_cars)}")
    print(f"Cars with no reachable station      : {len(infeasible_cars)}")
    if infeasible_cars:
        print("  ⚠️  Infeasible car IDs:", infeasible_cars)

    # Save reachability matrix
    reachability_matrix = np.zeros((N_CARS, m_stations), dtype=int)
    for (car_id, station_id) in reachability:
        reachability_matrix[car_id, station_id] = 1

    df_reachability = pd.DataFrame(
        reachability_matrix,
        columns=[f"station_{s['station_id']}_{s['nom']}" for s in stations],
    )
    df_reachability.insert(0, "car_id", range(N_CARS))
    reachability_output_path = os.path.join(
        OUTPUT_DIR, f"{output_prefix}_reachability_matrix_cars_stations.csv"
    )
    df_reachability.to_csv(reachability_output_path, index=False, encoding="utf-8-sig")

    if len(feasible_cars) == 0:
        print("  ⚠️  No feasible car in this scenario.")
        return {
            "scenario": output_prefix, "highway": selected_highway, "sens": selected_sens,
            "status": "NO_FEASIBLE_CAR", "n_cars": N_CARS, "feasible_cars": 0,
            "infeasible_cars": len(infeasible_cars), "assigned_cars": 0,
            "total_waiting_time_min": None, "average_waiting_time_min": None,
            "max_waiting_time_min": None,
        }

    # ── 4d. CP-SAT optimization model ─────────────────────────────────────────
    model  = cp_model.CpModel()
    solver = cp_model.CpSolver()

    # Binary assignment variables
    y = {
        (car["car_id"], station_id): model.NewBoolVar(f"y_car_{car['car_id']}_station_{station_id}")
        for car in feasible_cars
        for station_id in feasible_stations_by_car[car["car_id"]]
    }

    # Each feasible car must be assigned to exactly one station
    for car in feasible_cars:
        car_id = car["car_id"]
        model.Add(sum(y[(car_id, s)] for s in feasible_stations_by_car[car_id]) == 1)

    # ── Soft queue modeling ───────────────────────────────────────────────────
    station_slot_cars  = defaultdict(list)
    for (car_id, station_id), info in reachability.items():
        station_slot_cars[(station_id, info["slot"])].append(car_id)

    queue_vars      = []
    over_queue_vars = []
    load_vars       = {}

    for (station_id, slot), car_ids in station_slot_cars.items():
        nb_chargers = int(stations[station_id]["existing_chargers_model"])

        load  = model.NewIntVar(0, len(car_ids), f"load_station_{station_id}_slot_{slot}")
        q     = model.NewIntVar(0, len(car_ids), f"queue_station_{station_id}_slot_{slot}")
        over_q = model.NewIntVar(0, len(car_ids), f"over_queue_station_{station_id}_slot_{slot}")

        model.Add(
            load == sum(
                y[(cid, station_id)]
                for cid in car_ids
                if (cid, station_id) in y
            )
        )
        model.Add(q >= load - nb_chargers)
        model.Add(q >= 0)
        model.Add(over_q >= q - MAX_QUEUE_PER_STATION)
        model.Add(over_q >= 0)

        queue_vars.append(q)
        over_queue_vars.append(over_q)
        load_vars[(station_id, slot)] = load

    # ── Station load balancing ────────────────────────────────────────────────
    station_total_load = {}
    max_station_load   = model.NewIntVar(0, len(feasible_cars), "max_station_load")

    for station in stations:
        station_id = station["station_id"]
        possible_car_ids = [
            c["car_id"] for c in feasible_cars if (c["car_id"], station_id) in y
        ]
        station_total_load[station_id] = model.NewIntVar(
            0, len(feasible_cars), f"total_load_station_{station_id}"
        )
        model.Add(
            station_total_load[station_id] == sum(y[(cid, station_id)] for cid in possible_car_ids)
        )
        model.Add(max_station_load >= station_total_load[station_id])

    total_assigned_cars    = len(feasible_cars)
    average_load_scaled    = total_assigned_cars * 100 // m_stations
    deviation_vars         = []

    for station in stations:
        station_id  = station["station_id"]
        load_scaled = model.NewIntVar(0, total_assigned_cars * 100, f"load_scaled_station_{station_id}")
        dev         = model.NewIntVar(0, total_assigned_cars * 100, f"deviation_station_{station_id}")
        model.Add(load_scaled == station_total_load[station_id] * 100)
        model.AddAbsEquality(dev, load_scaled - average_load_scaled)
        deviation_vars.append(dev)

    # ── Early-stop penalty ────────────────────────────────────────────────────
    early_stop_terms = []
    for (car_id, station_id), var in y.items():
        feasible_list     = feasible_stations_by_car[car_id]
        station_rank      = feasible_list.index(station_id)
        last_rank         = len(feasible_list) - 1
        early_stop_penalty = last_rank - station_rank
        early_stop_terms.append(early_stop_penalty * var)

    # ── Weighted objective ────────────────────────────────────────────────────
    OVER_QUEUE_WEIGHT        = 100_000
    QUEUE_WEIGHT             = 10_000
    BALANCE_DEVIATION_WEIGHT = 500
    MAX_LOAD_WEIGHT          = 100
    EARLY_STOP_WEIGHT        = 1

    model.Minimize(
        OVER_QUEUE_WEIGHT        * sum(over_queue_vars)
        + QUEUE_WEIGHT           * sum(queue_vars)
        + BALANCE_DEVIATION_WEIGHT * sum(deviation_vars)
        + MAX_LOAD_WEIGHT        * max_station_load
        + EARLY_STOP_WEIGHT      * sum(early_stop_terms)
    )

    solver.parameters.max_time_in_seconds   = SOLVER_TIME_LIMIT_S
    solver.parameters.log_search_progress   = False

    status = solver.Solve(model)
    status_labels = {
        cp_model.OPTIMAL:    "OPTIMAL",
        cp_model.FEASIBLE:   "FEASIBLE",
        cp_model.INFEASIBLE: "INFEASIBLE",
        cp_model.UNKNOWN:    "UNKNOWN",
    }
    status_label = status_labels.get(status, "UNKNOWN")

    print(f"\nSolver status : {status_label}")
    print(f"Wall time     : {solver.WallTime():.3f} s")

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print("  No usable assignment found.")
        return {
            "scenario": output_prefix, "highway": selected_highway, "sens": selected_sens,
            "status": status_label, "n_cars": N_CARS,
            "feasible_cars": len(feasible_cars), "infeasible_cars": len(infeasible_cars),
            "assigned_cars": 0,
            "total_waiting_time_min": None, "average_waiting_time_min": None,
            "max_waiting_time_min": None,
        }

    # ── Extract assignments ───────────────────────────────────────────────────
    assignments = []
    for car in feasible_cars:
        car_id             = car["car_id"]
        assigned_station_id = None
        for station_id in feasible_stations_by_car[car_id]:
            if solver.Value(y[(car_id, station_id)]) == 1:
                assigned_station_id = station_id
                break
        if assigned_station_id is None:
            continue
        station = stations[assigned_station_id]
        info    = reachability[(car_id, assigned_station_id)]
        assignments.append({
            "scenario":                 output_prefix,
            "car_id":                   car_id,
            "highway":                  selected_highway,
            "sens":                     selected_sens,
            "entry_time_min":           car["entry_time_min"],
            "battery_pct":              car["battery_pct"],
            "remaining_range_km":       car["remaining_range_km"],
            "assigned_station_id":      assigned_station_id,
            "assigned_station_name":    station["nom"],
            "station_pk":               station["pk"],
            "station_chargers":         station["existing_chargers_model"],
            "arrival_time_min":         info["arrival_time_min"],
            "arrival_slot":             info["slot"],
            "battery_slack_km":         info["battery_slack_km"],
            "distance_to_station_km":   info["distance_to_station_km"],
            "distance_station_to_exit_km": info["distance_station_to_exit_km"],
        })

    df_assignments = pd.DataFrame(assignments)

    # ── 4e. Simulate actual waiting times (FIFO queue) ────────────────────────
    df_assignments["waiting_time_min"]  = 0.0
    df_assignments["charging_start_min"] = 0.0
    df_assignments["charging_end_min"]   = 0.0

    for station_id, group in df_assignments.groupby("assigned_station_id"):
        station     = stations[int(station_id)]
        nb_chargers = int(station["existing_chargers_model"])
        charger_available_times = [0.0] * nb_chargers

        for idx, row in group.sort_values("arrival_time_min").iterrows():
            arrival           = float(row["arrival_time_min"])
            next_idx          = int(np.argmin(charger_available_times))
            charging_start    = max(arrival, charger_available_times[next_idx])
            waiting_time      = charging_start - arrival
            charging_end      = charging_start + RECHARGE_TIME_MIN
            charger_available_times[next_idx] = charging_end

            df_assignments.loc[idx, "waiting_time_min"]   = waiting_time
            df_assignments.loc[idx, "charging_start_min"] = charging_start
            df_assignments.loc[idx, "charging_end_min"]   = charging_end

    total_waiting_time   = df_assignments["waiting_time_min"].sum()
    average_waiting_time = df_assignments["waiting_time_min"].mean()
    max_waiting_time     = df_assignments["waiting_time_min"].max()

    print(f"\nTotal waiting time   : {total_waiting_time:.2f} min")
    print(f"Average waiting time : {average_waiting_time:.2f} min")
    print(f"Max waiting time     : {max_waiting_time:.2f} min")

    print("\nFirst 20 assignments:")
    print(df_assignments.head(20).to_string(index=False))

    # ── Station summary ───────────────────────────────────────────────────────
    station_summary_rows = []
    for station in stations:
        station_id = station["station_id"]
        sub        = df_assignments[df_assignments["assigned_station_id"] == station_id]
        station_summary_rows.append({
            "scenario":                output_prefix,
            "highway":                 selected_highway,
            "sens":                    selected_sens,
            "station_id":              station_id,
            "station_name":            station["nom"],
            "pk":                      station["pk"],
            "chargers":                station["existing_chargers_model"],
            "assigned_cars":           len(sub),
            "total_waiting_time_min":  sub["waiting_time_min"].sum(),
            "average_waiting_time_min": sub["waiting_time_min"].mean() if len(sub) > 0 else 0.0,
            "max_waiting_time_min":    sub["waiting_time_min"].max()  if len(sub) > 0 else 0.0,
            "providers":               station.get("ev_providers", ""),
        })

    df_station_summary = pd.DataFrame(station_summary_rows)
    print("\nStation-level summary:")
    print(df_station_summary.to_string(index=False))

    # ── Decision matrix ───────────────────────────────────────────────────────
    decision_matrix = np.zeros((N_CARS, m_stations), dtype=int)
    for _, row in df_assignments.iterrows():
        decision_matrix[int(row["car_id"]), int(row["assigned_station_id"])] = 1

    df_decision_matrix = pd.DataFrame(
        decision_matrix,
        columns=[f"station_{s['station_id']}_{s['nom']}" for s in stations],
    )
    df_decision_matrix.insert(0, "car_id", range(N_CARS))

    # ── 4f. Save output files ─────────────────────────────────────────────────
    assignments_output_path    = os.path.join(OUTPUT_DIR, f"{output_prefix}_optimized_car_station_assignments.csv")
    summary_output_path        = os.path.join(OUTPUT_DIR, f"{output_prefix}_station_queue_summary.csv")
    decision_matrix_output_path = os.path.join(OUTPUT_DIR, f"{output_prefix}_decision_matrix_car_station.csv")

    df_assignments.to_csv(assignments_output_path,     index=False, encoding="utf-8-sig")
    df_station_summary.to_csv(summary_output_path,     index=False, encoding="utf-8-sig")
    df_decision_matrix.to_csv(decision_matrix_output_path, index=False, encoding="utf-8-sig")

    print("\n✅ Files saved:")
    print(f"   {assignments_output_path}")
    print(f"   {summary_output_path}")
    print(f"   {decision_matrix_output_path}")
    print(f"   {reachability_output_path}")

    return {
        "scenario":                 output_prefix,
        "highway":                  selected_highway,
        "sens":                     selected_sens,
        "status":                   status_label,
        "n_cars":                   N_CARS,
        "feasible_cars":            len(feasible_cars),
        "infeasible_cars":          len(infeasible_cars),
        "assigned_cars":            len(df_assignments),
        "n_stations":               m_stations,
        "total_waiting_time_min":   total_waiting_time,
        "average_waiting_time_min": average_waiting_time,
        "max_waiting_time_min":     max_waiting_time,
    }

In [8]:
# ============================================================
# STEP 5 bis — Find maximum N_CARS before queue condition fails
# ============================================================

import contextlib
import io
import gc

def compute_max_actual_queue(assignments_csv_path):
    """
    Computes the maximum actual number of cars waiting at the same time.
    A car is considered waiting between arrival_time_min and charging_start_min.
    """

    df = pd.read_csv(assignments_csv_path)

    if df.empty:
        return 0

    max_queue_global = 0

    for station_id, group in df.groupby("assigned_station_id"):

        events = []

        for _, row in group.iterrows():
            arrival = float(row["arrival_time_min"])
            start = float(row["charging_start_min"])

            # The car only waits if charging starts after arrival
            if start > arrival:
                events.append((arrival, +1))
                events.append((start, -1))

        # At the same time, remove cars from queue before adding new ones
        events.sort(key=lambda x: (x[0], x[1]))

        current_queue = 0
        max_queue_station = 0

        for _, change in events:
            current_queue += change
            max_queue_station = max(max_queue_station, current_queue)

        max_queue_global = max(max_queue_global, max_queue_station)

    return int(max_queue_global)


def test_n_cars(n_cars, verbose=False):
    """
    Runs all scenarios for a given number of cars and checks whether
    the max actual queue condition is respected.
    """

    global N_CARS
    N_CARS = n_cars

    scenario_results = []
    max_queue_global = 0
    all_ok = True

    for scenario_id, (highway, sens) in enumerate(SCENARIOS):

        if verbose:
            result = run_scenario(highway, sens, scenario_id)
        else:
            with contextlib.redirect_stdout(io.StringIO()):
                result = run_scenario(highway, sens, scenario_id)

        sens_label = "sens_plus" if sens == 1 else "sens_minus"
        output_prefix = f"{highway}_{sens_label}"

        assignments_path = os.path.join(
            OUTPUT_DIR,
            f"{output_prefix}_optimized_car_station_assignments.csv"
        )

        if os.path.exists(assignments_path):
            max_queue = compute_max_actual_queue(assignments_path)
        else:
            max_queue = None
            all_ok = False

        if max_queue is not None:
            max_queue_global = max(max_queue_global, max_queue)

            if max_queue > MAX_QUEUE_PER_STATION:
                all_ok = False

        if result["status"] not in ["OPTIMAL", "FEASIBLE"]:
            all_ok = False

        scenario_results.append({
            "n_cars": n_cars,
            "scenario": output_prefix,
            "status": result["status"],
            "assigned_cars": result["assigned_cars"],
            "infeasible_cars": result["infeasible_cars"],
            "average_waiting_time_min": result["average_waiting_time_min"],
            "max_waiting_time_min": result["max_waiting_time_min"],
            "max_actual_queue_observed": max_queue,
            "queue_condition_ok": max_queue is not None and max_queue <= MAX_QUEUE_PER_STATION,
        })

        gc.collect()

    return {
        "n_cars": n_cars,
        "feasible_condition": all_ok,
        "max_actual_queue_observed": max_queue_global,
        "details": scenario_results,
    }


def find_max_n_cars(n_min=10, n_max=500, step=10):
    """
    Tests N_CARS progressively and stops when the actual queue condition is violated.
    """

    summary_rows = []
    detail_rows = []
    best_n = None

    for n in range(n_min, n_max + 1, step):
        print(f"Testing N_CARS = {n}...")

        result = test_n_cars(n, verbose=False)

        summary_rows.append({
            "n_cars": n,
            "condition_respected": result["feasible_condition"],
            "max_actual_queue_observed": result["max_actual_queue_observed"],
        })

        for row in result["details"]:
            detail_rows.append(row)

        if result["feasible_condition"]:
            best_n = n
        else:
            print(f"Condition violated at N_CARS = {n}")
            break

    df_capacity_summary = pd.DataFrame(summary_rows)
    df_capacity_details = pd.DataFrame(detail_rows)

    summary_output_path = os.path.join(
        OUTPUT_DIR,
        "capacity_search_summary.csv"
    )

    details_output_path = os.path.join(
        OUTPUT_DIR,
        "capacity_search_details_by_scenario.csv"
    )

    df_capacity_summary.to_csv(
        summary_output_path,
        index=False,
        encoding="utf-8-sig"
    )

    df_capacity_details.to_csv(
        details_output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n" + "=" * 70)
    print("CAPACITY SEARCH SUMMARY")
    print("=" * 70)
    print(df_capacity_summary.to_string(index=False))

    if best_n is not None:
        print(f"\nMaximum tested N_CARS respecting the condition: {best_n}")
    else:
        print("\nNo tested value respected the condition.")

    print(f"\nSaved summary to: {summary_output_path}")
    print(f"Saved details to: {details_output_path}")

    return best_n, df_capacity_summary

In [9]:
# 3. Lancement

print("=" * 60)

print("STEP 5 — Finding maximum number of cars")

best_n, df_capacity_summary = find_max_n_cars(

    n_min=10,

    n_max=500,

    step=10

)

STEP 5 — Finding maximum number of cars
Testing N_CARS = 10...
Condition violated at N_CARS = 10

CAPACITY SEARCH SUMMARY
 n_cars  condition_respected  max_actual_queue_observed
     10                False                          5

No tested value respected the condition.

Saved summary to: Outputs/capacity_search_summary.csv
Saved details to: Outputs/capacity_search_details_by_scenario.csv


---
## 8. Step 5 — Run All Scenarios

We now execute `run_scenario()` for all six highway/direction combinations defined in `SCENARIOS`. Each call is independent and uses a slightly different random seed to diversify the EV population.

Results are collected into `global_summary` and then exported as `Outputs/global_scenario_summary.csv`.

The final summary table compares all six scenarios on:
- solver status (OPTIMAL / FEASIBLE / ...)
- number of feasible and assigned cars
- total, average, and maximum waiting time per car

In [10]:
print("=" * 60)
print("STEP 5 — Running all scenarios")
print("=" * 60)

global_summary = []

for scenario_id, (highway, sens) in enumerate(SCENARIOS):
    scenario_result = run_scenario(
        selected_highway=highway,
        selected_sens=sens,
        scenario_id=scenario_id,
    )
    global_summary.append(scenario_result)

df_global_summary = pd.DataFrame(global_summary)

global_summary_path = os.path.join(OUTPUT_DIR, "global_scenario_summary.csv")
df_global_summary.to_csv(global_summary_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 80)
print("GLOBAL SCENARIO SUMMARY")
print("=" * 80)
print(df_global_summary.to_string(index=False))
print(f"\n✅ Global summary saved: {global_summary_path}")

STEP 5 — Running all scenarios

################################################################################
SCENARIO 0 — A2, direction=1
################################################################################

Existing stations retained: 3
 station_id autoroute             nom  sens    pk  existing_chargers_model ev_providers
          0        A2    Teufengraben     1  40.0                        1             
          1        A2 Neuenkirch-West     1  86.0                        1             
          2        A2 Bellinzona-Nord     1 233.0                        4         Move

Entry PK     : 12.0 km
Exit PK      : 442.0 km
Axis length  : 430.0 km

First 10 generated cars:
 car_id  entry_time_min  entry_pk  exit_pk  battery_pct  remaining_range_km
      0               0      12.0    442.0     0.554849          221.939630
      1               1      12.0    442.0     0.312803          125.121144
      2               2      12.0    442.0     0.635081          254

---
## Summary

This notebook has:

1. **Loaded** six highway rest-area files and enriched them with known EV charger data.
2. **Generated** synthetic EV populations (100 cars per scenario) with randomized battery levels.
3. **Computed** reachability matrices to identify which stations each car can physically reach.
4. **Optimized** the centralized assignment of each EV to a charging station using Google OR-Tools CP-SAT, balancing queue lengths and station loads.
5. **Simulated** actual waiting times per car using a FIFO queue model.
6. **Exported** all results to the `Outputs/` folder.

All six scenarios (A2, A8, A13 — both directions) are summarized in `Outputs/global_scenario_summary.csv`.